# Baseline models

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
import numpy as np
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
import joblib

In [ ]:
df = pd.read_csv('../data/clean/full_data.csv')

In [ ]:
df.head()

In [ ]:
df.drop(columns=['casos'], inplace=True)
df.drop(columns = ["rr"], inplace = True)
df.drop(columns = ["clasi_rr"], inplace = True)
df.drop(columns = ["tmax_mean"], inplace = True)
df.drop(columns = ["tmin_mean"], inplace = True)
df.drop(columns = ["temp_prom"], inplace = True)
df.drop(columns = ["precip_total"], inplace = True)
df.drop(columns = ["precip_max"], inplace = True)
df.drop(columns = ["precip_min"], inplace = True)
df.drop(columns = ["precip_median"], inplace = True)
df.drop(columns = ["precip_mean"], inplace = True)
df.drop(columns = ["ndvi"], inplace = True)

In [ ]:
df.drop(columns = ["rr_lag_1"], inplace = True)
df.drop(columns = ["casos_lag_1"], inplace = True)
df.drop(columns = ["clasi_rr_lag_1"], inplace = True)
df.drop(columns = ["rr_lag_2"], inplace = True)
df.drop(columns = ["casos_lag_2"], inplace = True)
df.drop(columns = ["clasi_rr_lag_2"], inplace = True)
df.drop(columns = ["rr_lag_3"], inplace = True)
df.drop(columns = ["casos_lag_3"], inplace = True)
df.drop(columns = ["clasi_rr_lag_3"], inplace = True)
df.drop(columns = ["rr_lag_4"], inplace = True)
df.drop(columns = ["casos_lag_4"], inplace = True)
df.drop(columns = ["clasi_rr_lag_4"], inplace = True)
df.drop(columns = ["rr_lag_5"], inplace = True)
df.drop(columns = ["casos_lag_5"], inplace = True)
df.drop(columns = ["clasi_rr_lag_5"], inplace = True)
df.drop(columns = ["rr_lag_6"], inplace = True)
df.drop(columns = ["casos_lag_6"], inplace = True)
df.drop(columns = ["clasi_rr_lag_6"], inplace = True)
df.drop(columns = ["rr_lag_7"], inplace = True)
df.drop(columns = ["casos_lag_7"], inplace = True)
df.drop(columns = ["clasi_rr_lag_7"], inplace = True)
df.drop(columns = ["rr_lag_8"], inplace = True)
df.drop(columns = ["casos_lag_8"], inplace = True)
df.drop(columns = ["clasi_rr_lag_8"], inplace = True)

In [ ]:
df.shape

In [ ]:
df = pd.get_dummies(df, columns = ["urb"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_1"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_2"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_3"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_4"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_5"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_6"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_7"])
df = pd.get_dummies(df, columns = ["clasi_rr_lag_8"])

In [ ]:
df.head()

In [ ]:
train = df[df['week_canton'] < '2024-1-101']
val = df[(df['week_canton'] >= '2024-1-101') & (df['week_canton'] < '2025-1-101')]
test = df[df['week_canton'] >= '2025-1-101']

X_train = train.drop(columns = ["clasi_rr", "week_canton"])
y_train = train["clasi_rr"]

X_val = val.drop(columns = ["clasi_rr", "week_canton"])
y_val= val["clasi_rr"]

X_test= test.drop(columns = ["clasi_rr", "week_canton"])
y_test = test["clasi_rr"]

In [ ]:
scaler = StandardScaler()  

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)  
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
X_test_for_reg = pd.concat([X_val, X_test], axis=0)
y_test_for_reg = pd.concat([y_val, y_test], axis=0)

X_test_for_reg_scaled = pd.DataFrame(scaler.transform(X_test_for_reg), columns=X_test_for_reg.columns)

In [ ]:
train.shape, val.shape, test.shape

## Logistic regression

In [ ]:
reg = LogisticRegression()

reg.fit(X_train_scaled, y_train)

y_pred_reg = reg.predict(X_test_for_reg_scaled)

In [ ]:
print(classification_report(y_test_for_reg, y_pred_reg))

In [ ]:
y_score_reg = reg.predict_proba(X_test_for_reg)

label_binarizer = LabelBinarizer().fit(y_train)
y_onehot_test = label_binarizer.transform(y_test_for_reg)

In [ ]:
def roc_auc_metrics_log(class_of_interest, other_classes):
    class_id = np.flatnonzero(label_binarizer.classes_ == class_of_interest)[0]

    display = RocCurveDisplay.from_predictions(
        y_onehot_test[:, class_id],
        y_score_reg[:, class_id],
        name=f"{class_of_interest} contra el resto",
        curve_kwargs=dict(color="darkorange"),
        plot_chance_level=True,
        despine=True,
    )
    _ = display.ax_.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"One-vs-Rest ROC curves:\n{class_of_interest} vs ({other_classes[0]} and {other_classes[1]})",
    )

In [ ]:
roc_auc_metrics_log("Alto", ["Bajo", "Medio"])

In [ ]:
roc_auc_metrics_log("Bajo", ["Alto", "Medio"])

In [ ]:
roc_auc_metrics_log("Medio", ["Bajo", "Alto"])

In [ ]:
cnf_matrix_reg = confusion_matrix(y_test_for_reg, y_pred_reg)
cnf_matrix_reg

In [ ]:
clasi = ["Alto", "Bajo", "Medio"]
ConfusionMatrixDisplay(confusion_matrix=cnf_matrix_reg, display_labels = clasi).plot()

In [ ]:
coef_df = pd.DataFrame({"Feature": X_train.columns, "Coefficient": reg.coef_[0]})
sd = pd.DataFrame({"Feature": X_train.columns, "Std": X_train_scaled.std()})
coef_df = coef_df.merge(sd, on = "Feature")
coef_df["Coefficient_std"] = coef_df["Coefficient"] * coef_df["Std"]
coef_reg_sorted = coef_df.sort_values(by = "Coefficient_std", ascending=False)

coef_reg_sorted

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
coef_reg_sorted = coef_df.sort_values(by = "Coefficient_std", ascending=True)
ax.barh(coef_reg_sorted["Feature"].tail(10), coef_reg_sorted["Coefficient_std"].tail(10))
ax.bar_label(ax.containers[0], fmt='%.2f')
ax.set_xlabel("Coefficient")

### Random forest

In [ ]:
rf = RandomForestClassifier(oob_score=True)

param_grid_rf = {
    'max_depth': [5, 6, 7, 8],
    'min_samples_split': [10, 100, 500],
    'ccp_alpha': [0, 1 /  10**5, 1 / 10**4, 1 / 10**3, 0.01, 0.1, 1, 10], 
    "criterion": ["gini", "entropy"]
}

# 3. Initialize and run Grid Search
grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv = 5, n_jobs=-1, verbose=10)
grid_rf.fit(X_train, y_train)

In [ ]:
joblib.dump(grid_rf, "../models/clasi_n0_0_rf.pkl")

In [ ]:
y_pred_rf = grid_rf.predict(X_test_for_reg)

In [ ]:
y_score_rf = grid_rf.predict_proba(X_test_for_reg)

label_binarizer = LabelBinarizer().fit(y_train)
y_onehot_test = label_binarizer.transform(y_test_for_reg)

In [ ]:
def roc_auc_metrics_rf(class_of_interest, other_classes):
    class_id = np.flatnonzero(label_binarizer.classes_ == class_of_interest)[0]

    display = RocCurveDisplay.from_predictions(
        y_onehot_test[:, class_id],
        y_score_rf[:, class_id],
        name=f"{class_of_interest} contra el resto",
        curve_kwargs=dict(color="darkorange"),
        plot_chance_level=True,
        despine=True,
    )
    _ = display.ax_.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"One-vs-Rest ROC curves:\n{class_of_interest} vs ({other_classes[0]} and {other_classes[1]})",
    )

In [ ]:
roc_auc_metrics_rf("Alto", ["Bajo", "Medio"])

In [ ]:
roc_auc_metrics_rf("Alto", ["Bajo", "Medio"])

In [ ]:
roc_auc_metrics_rf("Alto", ["Bajo", "Medio"])

In [ ]:
print(classification_report(y_test_for_reg, y_pred_rf))

In [ ]:
cnf_matrix_rf = confusion_matrix(y_test_for_reg, y_pred_rf)
cnf_matrix_rf

In [ ]:
ConfusionMatrixDisplay(confusion_matrix=cnf_matrix_rf).plot()

In [ ]:
imp_rf_df = pd.DataFrame({"Feature": X_train.columns, "Importance": grid_rf.best_estimator_.feature_importances_})
imp_rf_sorted = imp_rf_df.sort_values(by = "Importance", ascending=False)

imp_rf_sorted

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
imp_rf_sorted = imp_rf_df.sort_values(by = "Importance", ascending=True)
ax.barh(imp_rf_sorted["Feature"].tail(10), imp_rf_sorted["Importance"].tail(10))
ax.bar_label(ax.containers[0], fmt='%.2f')
ax.set_xlabel("Importance")

## XGBoost

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test_for_reg)

In [ ]:
xgb = XGBClassifier()

param_grid_xgb = {
    'max_depth': [5, 6, 7, 8],
    'learning_rate': [0.3, 0.1, 0.05],
    'n_estimator': [50, 100, 150], 
    "criterion": ["auc", "logloss"],
    "reg_lambda": [0, 1 /  10**5, 1 / 10**4, 1 / 10**3, 0.01, 0.1, 1, 10],
}

grid_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv = 5, n_jobs=-1, verbose=0)
grid_xgb.fit(X_train, y_train_encoded)

In [ ]:
joblib.dump(grid_xgb, "../models/clasi_n0_0_xgb.pkl")

In [ ]:
y_pred_xgb = grid_xgb.predict(X_test_for_reg)

In [ ]:
y_score_xgb = grid_xgb.predict_proba(X_test_for_reg)

label_binarizer = LabelBinarizer().fit(y_train_encoded)
y_onehot_test = label_binarizer.transform(y_test_encoded)

In [ ]:
classes = ["Alto", "Bajo", "Medio"]
def roc_auc_metrics_xgb(class_of_interest, other_classes):
    class_id = np.flatnonzero(label_binarizer.classes_ == class_of_interest)[0]

    display = RocCurveDisplay.from_predictions(
        y_onehot_test[:, class_id],
        y_score_xgb[:, class_id],
        name=f"{classes[class_of_interest]} contra el resto",
        curve_kwargs=dict(color="darkorange"),
        plot_chance_level=True,
        despine=True,
    )
    _ = display.ax_.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"One-vs-Rest ROC curves:\n{classes[class_of_interest]} vs ({classes[other_classes[0]]} and {classes[other_classes[1]]})",
    )

In [ ]:
roc_auc_metrics_xgb(0, [1, 2])

In [ ]:
roc_auc_metrics_xgb(1, [0, 2])

In [ ]:
roc_auc_metrics_xgb(2, [1, 0])

In [ ]:
print(classification_report(y_test_encoded, y_pred_xgb))

In [ ]:
cnf_matrix_xgb = confusion_matrix(y_test_encoded, y_pred_xgb)
cnf_matrix_xgb

In [ ]:
ConfusionMatrixDisplay(confusion_matrix=cnf_matrix_xgb).plot()

In [ ]:
imp_xgb_df = pd.DataFrame({"Feature": X_train.columns, "Importance": grid_xgb.best_estimator_.feature_importances_})
imp_xgb_sorted = imp_xgb_df.sort_values(by = "Importance", ascending=False)

imp_xgb_sorted

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
imp_xgb_sorted = imp_xgb_df.sort_values(by = "Importance", ascending=True)
ax.barh(imp_xgb_sorted["Feature"].tail(10), imp_xgb_sorted["Importance"].tail(10))
ax.bar_label(ax.containers[0], fmt='%.2f')
ax.set_xlabel("Importance")

In [ ]:
# REMEMBER TO RUN THE CODE WITHOUT URB AND CLASSI DUMMIES

In [ ]:
# y_encoded_xgb = OrdinalEncoder().fit_transform(df[["clasi_rr"]])

# df["urb"] = df["urb"].astype("category")
# df["clasi_rr_lag_1"] = df["clasi_rr_lag_1"].astype("category")
# df["clasi_rr_lag_2"] = df["clasi_rr_lag_2"].astype("category")
# df["clasi_rr_lag_3"] = df["clasi_rr_lag_3"].astype("category")
# df["clasi_rr_lag_4"] = df["clasi_rr_lag_4"].astype("category")
# df["clasi_rr_lag_5"] = df["clasi_rr_lag_5"].astype("category")
# df["clasi_rr_lag_6"] = df["clasi_rr_lag_6"].astype("category")
# df["clasi_rr_lag_7"] = df["clasi_rr_lag_7"].astype("category")
# df["clasi_rr_lag_8"] = df["clasi_rr_lag_8"].astype("category")

# train = df[df['week_canton'] < '2024-1-101']
# val = df[(df['week_canton'] >= '2024-1-101') & (df['week_canton'] < '2025-1-101')]
# test = df[df['week_canton'] >= '2025-1-101']

# X_train = train.drop(columns = ["clasi_rr", "week_canton"])
# y_train = y_encoded_xgb[train.index].ravel()

# X_val = val.drop(columns = ["clasi_rr", "week_canton"])
# y_val = y_encoded_xgb[val.index].ravel()

# X_test= test.drop(columns = ["clasi_rr", "week_canton"])
# y_test = y_encoded_xgb[test.index].ravel()

In [ ]:
# dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
# dval = xgb.DMatrix(X_val, y_val, enable_categorical=True)
# dtest = xgb.DMatrix(X_test, y_test, enable_categorical=True)

In [ ]:
# params = {"objective": "multi:softprob", "tree_method": "hist", "num_class": 3, "eval_metric": ["auc"]}

# n = 100

# xgb_model = xgb.train(
#    params=params,
#    dtrain=dtrain,
#    num_boost_round=n,
#    evals=[(dtrain, "train"), (dval, "validation")],
#    verbose_eval=10, 
#    early_stopping_rounds=50
# )

In [ ]:
# xgb_model.best_score

In [ ]:
# import numpy as np
# y_pred_proba = xgb_model.predict(dtest)
# y_pred = np.argmax(y_pred_proba, axis=1)

In [ ]:
# print(classification_report(y_test, y_pred))

In [ ]:
# cnf_matrix_xgb = confusion_matrix(y_test, y_pred)

In [ ]:
# ConfusionMatrixDisplay(confusion_matrix=cnf_matrix_xgb).plot()

In [ ]:
# importance_xgb = xgb_model.get_score(importance_type="weight")
# importance_xgb_df = pd.DataFrame({"Feature": list(importance_xgb.keys()), "Importance": list(importance_xgb.values())})
# importance_xgb_sorted = importance_xgb_df.sort_values(by = "Importance", ascending=False) 
# importance_xgb_sorted.head(10)

In [ ]:
# xgb.plot_importance(xgb_model, max_num_features=10, importance_type="weight")